# Checkpoints

## 1. `torch.save()`

Serializes a Python object and saves it to disk.

**Serialization** is the process of converting an object (such as a dictionary, tensor, or number) into a representation that can be stored and reconstructed later.

- **Not inherently atomic:** an interrupted write can leave an incomplete checkpoint. A common solution is to save to a temporary file (e.g., `.partial`) and then use `os.replace()` to replace the destination.
- **Does not automatically save your model's source code or architecture:** when saving a `state_dict`, you must recreate the model architecture before loading its weights.
- **Can save more than model weights:** for example, optimizer states, training steps, loss values, and other supported Python objects.

```python
# Save a Python object to disk
torch.save(obj, path)
```

## 2. `torch.load()`

Loads an object previously serialized in a format compatible with PyTorch, typically using `torch.save()`.

It reconstructs the saved object in memory. The `map_location` argument specifies where tensors should be loaded, allowing you to load GPU checkpoints onto the CPU, for example.

```python
# Load the checkpoint onto the CPU
checkpoint = torch.load(
    "checkpoint.pth",
    map_location="cpu",
    weights_only=True
)
```

- `map_location="cpu"`: loads tensors onto the CPU, regardless of their original device.
- `weights_only=True`: restricts loading to tensors and supported types, reducing the risks associated with untrusted checkpoints.

**Important:** `torch.load()` is not a universal file reader. It is intended for compatible serialized objects, not arbitrary files such as CSV, JSON, or raw binary datasets.

## 3. `load_state_dict()`

Loads a model's parameters and persistent buffers from a state dictionary into an already instantiated model.

Unlike `torch.load()`, which reconstructs the saved object, `load_state_dict()` transfers the saved state into an existing model instance.

```python
# Recreate the model architecture
model = MyModel()

# Load the saved weights
state_dict = torch.load(
    "weights.pth",
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict)
```

**Key distinction:**

- `torch.save()` → serializes and saves an object.
- `torch.load()` → loads and reconstructs a serialized object.
- `model.load_state_dict()` → applies the saved parameters and buffers to an existing model.

For training checkpoints, you may also need to restore the optimizer state, training step, scheduler state, and other information required to resume training.

In [ ]:
import os, re
from glob import glob
import torch


def save_checkpoints(path, step, model, optimizer, scheduler, loss, total_tokens, keep_last = 3):
    tmp_path = f'{path}/checkpoint_{step}.partial'
    checkpoint_path = f'{path}/checkpoint_{step}.pt'

    try:
        # Save the model partially
        torch.save({
            'global_step': step,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'loss': loss,
            'total_tokens': total_tokens
        }, tmp_path)

        # Change the type file (.partil to .pt) after all the bytes of states are ready
        os.replace(tmp_path, checkpoint_path)
        print(f'Checkpoint, step {step} salvo em {checkpoint_path}')

    # If something went wrong, remove the last checkpoint
    except BaseException:
        print(f'Ocorreu um erro ao salvar o modelo! Excluindo versão antiga!')
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise 

    # Get all the checkpoins after the save
    total_checkpoints = sorted(
        glob(f'{path}/checkpoint*.pt'),
        key=lambda p: int(re.search(r'checkpoint_(\d+)\.pt$', p).group(1)),)

    # Mantain the last KEEP_LAST checkpoints
    old = total_checkpoints[:-keep_last]
    for x in old:
        os.remove(x)
    if old:
        print(f'Checkpoint(s) antigo(s) removido(s): {old}')


def load_checkpoints(path, model, optimizer, scheduler):
    # See if we already have some checkpoints from last epochs 

    CKPT_RE = re.compile(r'checkpoint_(\d+)\.pt$')
    old_checkpoints = sorted(
        glob(f'{path}/checkpoint_*.pt'),
        key=lambda p: int(CKPT_RE.search(p).group(1)),
    )
    if old_checkpoints:
        # Get the last checkpoint and load their infos
        last = torch.load(old_checkpoints[-1], map_location='cpu')

        model.load_state_dict(last['model'])
        optimizer.load_state_dict(last['optimizer'])
        scheduler.load_state_dict(last['scheduler'])
       
        print(f'Checkpoint encontrado, tudo carregado e salvo')

        # Return in order: global_step, loss, total_tokens:
        return last['global_step'], last['loss'], last['total_tokens']
    else:
        print(f'Nenhum checkpoint antigo encontrado')

        # Return in order: global_step, loss, total_tokens:
        return 0, 0.0, 0 


# Create the path if doesnt exist, if exist just ignore (before the training)
path = 'path/path/path'
os.makedirs(path, exist_ok = True)

# Create the global variables
CHECKPOINT_SAVE_EVERY_STEPS = 5000
global_step = 0
loss = 0
total_tokens = 0

# If we have checkpoints load them
global_step, loss, total_tokens = load_checkpoints(path, model, optimizer, scheduler)

#  In the TRAINING LOOP
if (global_step + 1) % CHECKPOINT_SAVE_EVERY_STEPS == 0:
    save_checkpoints(path, global_step + 1, model, optimizer, scheduler, loss, total_tokens)
